In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)  # reproducibility

In [ ]:
def simulate_random_walk(N=20, L=50, steps=200, step_size=1.0,
                          collision_radius=1.0, cooldown=5):
    
    pos = np.random.uniform(0, L, size=(N, 2))
    positions_history = np.zeros((steps, N, 2))

    collision_count = 0
    collision_points = []
    # last_collision_step[i,j] tracks cooldown per pair
    last_collision_step = -np.ones((N, N)) * (cooldown + 1)

    for t in range(steps):
        # ---- RANDOM WALK STEP (no chasing, no targets) ----
        angles = np.random.uniform(0, 2 * np.pi, size=N)
        dx = step_size * np.cos(angles)
        dy = step_size * np.sin(angles)
        pos[:, 0] += dx
        pos[:, 1] += dy

        pos[:, 0] = np.clip(pos[:, 0], 0, L)
        pos[:, 1] = np.clip(pos[:, 1], 0, L)

        positions_history[t] = pos

        diff = pos[:, None, :] - pos[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=-1))

        for i in range(N):
            for j in range(i + 1, N):
                if dist[i, j] < collision_radius:
                    if t - last_collision_step[i, j] > cooldown:
                        collision_count += 1
                        collision_points.append((t, i, j, pos[i, 0], pos[i, 1]))
                        last_collision_step[i, j] = t

    return positions_history, collision_count, collision_points

In [ ]:
N_demo = 15
L_demo = 30
steps_demo = 150

hist, n_coll, coll_pts = simulate_random_walk(N=N_demo, L=L_demo, steps=steps_demo,
                                               step_size=1.0, collision_radius=1.2,
                                               cooldown=5)

plt.figure(figsize=(6, 6))
for agent in range(N_demo):
    plt.plot(hist[:, agent, 0], hist[:, agent, 1], alpha=0.6, linewidth=1)
    plt.scatter(hist[-1, agent, 0], hist[-1, agent, 1], s=25)

# mark collision locations
if coll_pts:
    cx = [p[3] for p in coll_pts]
    cy = [p[4] for p in coll_pts]
    plt.scatter(cx, cy, marker='x', color='red', s=80, label='Collision')

plt.title(f"Random Walk (L={L_demo}, N={N_demo})\nTotal collisions = {n_coll}")
plt.xlabel("X")
plt.ylabel("Y")
plt.xlim(0, L_demo)
plt.ylim(0, L_demo)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Area = {L_demo*L_demo}, Collisions = {n_coll}")

In [ ]:
N_agents      = 25
steps_exp     = 300
step_size_exp = 1.0
coll_radius   = 1.2
trials        = 5  

L_values = [10, 15, 20, 30, 40, 60, 80, 100, 130, 160, 200]

results = []
for L in L_values:
    coll_runs = []
    for trial in range(trials):
        _, n_coll, _ = simulate_random_walk(N=N_agents, L=L, steps=steps_exp,
                                             step_size=step_size_exp,
                                             collision_radius=coll_radius,
                                             cooldown=5)
        coll_runs.append(n_coll)
    avg_coll = np.mean(coll_runs)
    area = L * L
    results.append({"L": L, "Area": area, "Avg_Collisions": avg_coll,
                     "Min": np.min(coll_runs), "Max": np.max(coll_runs)})
    print(f"L={L:4d}  Area={area:8.0f}  Avg Collisions={avg_coll:6.2f}  (trials={coll_runs})")

df = pd.DataFrame(results)
df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df["Area"], df["Avg_Collisions"], marker='o', linewidth=2, color='crimson')
plt.title("Number of Collisions vs Area\n(Random-walk agents, no chasing)")
plt.xlabel("Area (L x L)")
plt.ylabel("Average Number of Collisions")
plt.grid(alpha=0.4)
plt.show()